In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import sys
import os
import pickle

import pickle

from scipy.spatial.distance import cdist
from PIL import Image
import os
import cv2
import random
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist,squareform
import networkx as nx
import random
from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import cdist
from sklearn.preprocessing import normalize
from scipy.spatial import cKDTree
from sklearn.metrics import mean_squared_error
from scipy.linalg import orthogonal_procrustes
from itertools import product
from tqdm import tqdm
from pathlib import Path
from matplotlib.image import imread
from sklearn.metrics.pairwise import euclidean_distances
import json
from sklearn.utils import shuffle

import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.metrics.pairwise import euclidean_distances
from scipy.spatial.distance import pdist, squareform
import os
import glob
import pandas as pd
import numpy as np
from pathlib import Path

# for Signals data
from data_analysis.io_f import read_data_file    # for reading .txt trace files  
from data_analysis.compute_f import split_ts_seq, compute_step_positions  
from data_analysis.visualize_f import visualize_heatmap  



In [3]:
EMBEDDING_DIM = 3
NUM_POINTS = 2000


# Functions

In [4]:
def knn_graph(matrix, k=5):
    G = nx.Graph()
    nbrs = NearestNeighbors(n_neighbors=k).fit(matrix)
    distances, indices = nbrs.kneighbors(matrix)
    
    for i in range(len(matrix)):
        G.add_node(i, pos=(matrix[i, 0], matrix[i, 1]))
        for j in indices[i]:
            if i != j:
                G.add_edge(i, j, weight=distances[i][np.where(indices[i] == j)[0][0]])
    
    return G

def plot_knn_graph(G,highlighted_nodes,polygon_mask):
    
    # Create a color map for nodes
    highlight_color = "#FF4500"
    node_colors = [highlight_color if node in highlighted_nodes else 'lightblue' for node in G.nodes()]
    #if pos == None:    
    pos = nx.get_node_attributes(G, 'pos')
    # Draw the graph
    fig,ax = plt.subplots(figsize=(8, 8))
    ax.imshow(polygon_mask, cmap='gray', alpha=0.3)
    nx.draw(G, pos, with_labels=False, node_size=100, node_color=node_colors, edge_color='gray', alpha=0.6,ax = ax)
    return fig,ax
    
def plot_scatter(pos,highlighted_nodes,polygon_mask):
    
    # Create a color map for nodes
    highlight_color = "#FF4500"
    node_colors = [highlight_color if i in highlighted_nodes else 'lightblue' for i in np.arange(1000)]
    #if pos == None:    
    #pos = nx.get_node_attributes(G, 'pos')
    # Draw the graph
    fig,ax = plt.subplots(figsize=(8, 8))
    ax.imshow(polygon_mask, cmap='gray', alpha=0.3)
    ax.scatter(pos[:,0],pos[:,1],s = 50,c = node_colors)
    #nx.draw(G, pos, with_labels=False, node_size=100, node_color=node_colors, edge_color='gray', alpha=0.6,ax = ax)
    return fig,ax


    #nx.draw(G, pos, with_labels=False, node_size=100, node_color='lightblue', edge_color='gray', alpha=0.6)
    #plt.show()

def plot_knn_graph_pos(G,highlighted_nodes,pos):
    #pos = nx.get_node_attributes(G, 'pos')
    highlight_color = "#FF4500"
    node_colors = [highlight_color if node in highlighted_nodes else 'lightblue' for node in G.nodes()]
    fig,ax = plt.subplots(figsize=(8, 8))
    nx.draw(G, pos, with_labels=False, node_size=100, node_color=node_colors, edge_color='gray', alpha=0.6,ax = ax)
    return fig,ax
    
def compute_kernel_matrix(X,k):
    D = squareform(pdist(X))
    D_s = np.argsort(D,axis=1)
    sig = np.median(D_s[:,(k+1)])
    # D_sorted = np.sort(D, axis=1) 
    # k_nn_distances = D_sorted[:, k] 
    # sig = np.median(k_nn_distances)
    print("SIG:", sig) 
    K = np.exp(-D**2/sig**2)    
    return K

def compute_robust_kernel(fingerprints, k=10):
    # 1. Use COSINE distance (1 - cosine_similarity)
    # robust to absolute signal strength differences
    dists = pdist(fingerprints, metric='cosine')
    D = squareform(dists)
    # 2. Adaptive Sigma (Self-Tuning)
    # sigma_i = distance to k-th neighbor
    sorted_dists = np.sort(D, axis=1)
    sigma = sorted_dists[:, k]  # (N,)
    # 3. Construct Kernel
    N = D.shape[0]
    K = np.zeros((N, N))
    # Vectorized calculation for speed
    # K_ij = exp( - dist^2 / (sigma_i * sigma_j) )
    # We use outer product to get matrix of sigma_i * sigma_j
    sig_prod = np.outer(sigma, sigma)
    # Avoid division by zero
    sig_prod[sig_prod < 1e-8] = 1.0 
    K = np.exp(- (D**2) / sig_prod)
    return K

def compute_laplacian(K):
    D_inv = np.diag(np.sum(K,axis = 1)**(-0.5))
    L = D_inv@K@D_inv
    return L

def compute_leading_eigenvectors(L, d):
    lam,v = np.linalg.eigh(L)
    sort_idx = np.argsort(lam)[::-1]
    lam = lam[sort_idx]
    v = v[:,sort_idx]
    return lam[:d],v[:,:d]

# def segment_inside_mask(mask, p0, p1, thickness=1, thresh=128):
#     """
#     Return True if the full line between p0->p1 lies inside mask>thresh.
#     p0, p1 are (x, y) coordinates.
#     """
#     H, W = mask.shape[:2]
#     tmp = np.zeros((H, W), np.uint8)
#     cv2.line(tmp, (int(p0[0]), int(p0[1])), (int(p1[0]), int(p1[1])), 255, thickness)
#     walkable = mask > thresh
#     return np.all(walkable[tmp > 0])

def remove_edges_outside_mask(G, mask, thickness=2, thresh=128, verbose=True):
    """
    Removes edges from G whose line segments leave the mask.
    Edits G in place.
    """
    pos = nx.get_node_attributes(G, "pos")
    to_remove = []
    
    for u, v in list(G.edges()):
        if u not in pos or v not in pos:
            to_remove.append((u, v))
            continue
        
        if not segment_inside_mask(mask, pos[u], pos[v], thickness=thickness, thresh=thresh):
            to_remove.append((u, v))
    
    G.remove_edges_from(to_remove)
    
    if verbose:
        print(f"❌ Removed {len(to_remove)} edges that crossed outside the mask "
              f"({len(G.edges())} remain)")
    return to_remove


    

In [5]:

def load_image(image_path):
    image = cv2.imread(image_path)
    image = cv2.resize(image, (image.shape[1] // 4, image.shape[0] // 4))  # Reduce size by half
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    return image, hsv

def detect_shapes(image):
    polygon_mask = np.zeros(image.shape[:2], dtype=np.uint8)
    rectangles = []
    circles = []
    
    # Define color ranges
    black_lower = np.array([0, 0, 0])
    black_upper = np.array([180, 255, 50])
    blue_lower = np.array([100, 150, 50])
    blue_upper = np.array([140, 255, 255])
    green_lower = np.array([40, 50, 50])
    green_upper = np.array([90, 255, 255])
    
    # Detect black polygon
    black_mask = cv2.inRange(image, black_lower, black_upper)
    contours, _ = cv2.findContours(black_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours:
        cv2.drawContours(polygon_mask, [contour], -1, 255, thickness=cv2.FILLED)
    
    # Detect blue rectangles
    blue_mask = cv2.inRange(image, blue_lower, blue_upper)
    contours, _ = cv2.findContours(blue_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        rectangles.append((x, y, w, h))
    
    # Detect green circles
    green_mask = cv2.inRange(image, green_lower, green_upper)
    contours, _ = cv2.findContours(green_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours:
        (x, y), radius = cv2.minEnclosingCircle(contour)
        circles.append((int(x), int(y), int(radius)))
    
    return polygon_mask, rectangles, circles


def generate_random_points(mask, num_points=1000):
    points = []
    h, w = mask.shape
    while len(points) < num_points:
        x, y = random.randint(0, w-1), random.randint(0, h-1)
        if mask[y, x] == 255:
            points.append((x, y))
    return points

# def build_graph(mask, rectangles):
#     G = nx.grid_2d_graph(mask.shape[0], mask.shape[1])
#     for x, y, w, h in rectangles:
#         for i in range(x, x+w):
#             for j in range(y, y+h):
#                 if (j, i) in G:                    
#                     G.remove_node((j, i))
#     return G


def build_graph(mask, rectangles=None):
    """
    Build a grid graph from a binary mask.

    Parameters
    ----------
    mask : np.ndarray (H x W)
        Binary mask (1 or 255 = inside / walkable area).
    rectangles : list of (x, y, w, h), optional
        Rectangles to remove as obstacles. If None, obstacles are taken
        from where mask == 0 (i.e., the inverse of mask).

    Returns
    -------
    G : networkx.Graph
        Grid graph with nodes removed for obstacle regions.
    """
    H, W = mask.shape[:2]
    G = nx.grid_2d_graph(H, W)

    # Case 1: explicit rectangles
    if rectangles is not None and len(rectangles) > 0:
        for x, y, w, h in rectangles:
            for i in range(x, x + w):
                for j in range(y, y + h):
                    if (j, i) in G:
                        G.remove_node((j, i))

    # Case 2: implicit obstacles from mask (0 = blocked)
    else:
        blocked = np.where(mask == 0)
        for j, i in zip(blocked[0], blocked[1]):  # note (row, col)
            if (j, i) in G:
                G.remove_node((j, i))
    

    return G

            
def segment_inside_mask(mask, p0, p1, thickness=2, thresh=128):
    H, W = mask.shape[:2]
    tmp = np.zeros((H, W), np.uint8)
    cv2.line(tmp, (int(p0[0]), int(p0[1])), (int(p1[0]), int(p1[1])), 255, thickness)
    walkable = mask > thresh
    return np.all(walkable[tmp > 0])


# def create_complete_graph_from_points(mask, points, width_m, height_m, obey_mask=True, thickness=2):
#     # ... (Keep the setup code: Standardizing input, H_pixels, W_pixels calculation) ...
#     # Ensure you have the 'pixels' array calculated correctly as before
    
#     # ... (Inside the loop) ...
#     for i in range(N):
#         px1, py1 = pixels[i]
#         x1, y1 = points_data[i]
        
#         for j in range(i+1, N):
#             px2, py2 = pixels[j]
#             x2, y2 = points_data[j]
            
#             # 1. Base Euclidean Distance
#             dist = float(np.hypot(x2 - x1, y2 - y1))
            
#             # 2. Connection Radius (e.g., 30m)
#             if dist > 30.0: 
#                 continue

#             # 3. SOFT WALL LOGIC
#             weight = dist
            
#             if obey_mask:
#                 # Check if line hits a wall
#                 is_clear = segment_inside_mask(mask, (px1, py1), (px2, py2), thickness=thickness)
                
#                 if not is_clear:
#                     # INSTEAD OF "continue" (Deleting), we PENALIZE
#                     # Walls attenuate signals. Let's say a wall makes the distance feel 2.5x longer.
#                     WALL_PENALTY = 2.5 
#                     weight = dist * WALL_PENALTY
            
#             # Add the edge with the (potentially penalized) weight
#             G.add_edge(i, j, weight=weight)
#             count_edges += 1
            
#     print(f"✅ Created Soft-Wall Graph: {G.number_of_nodes()} nodes, {count_edges} edges")
#     return G

def create_complete_graph_from_points(mask, points, width_m, height_m, obey_mask=True, thickness=2):
    """
    points: DataFrame or array with 'x', 'y' (in METERS)
    mask: 2D image (in PIXELS)
    width_meter, height_meter: Real world dimensions of the map
    """
    print("width", width_m)
    print("height", height_m)
    # --- 1. Handle Input Data (Standardize to Array) ---
    if isinstance(points, pd.DataFrame):
        # Ensure we get [x, y] order
        points_data = points[['x', 'y']].values
    else:
        points_data = points

    G = nx.Graph()
    N = len(points_data)
    
    # Image dimensions (H, W)
    H_pixels, W_pixels = mask.shape
    
    # --- 2. Add Nodes (Store positions in METERS) ---
    for i, (x, y) in enumerate(points_data):
        G.add_node(i, pos=(float(x), float(y)))

    if N == 0:
        return G

    # --- 3. Pre-calculate Pixel Coordinates for All Nodes ---
    # We need pixels ONLY for the wall check.
    # Formula: 
    #   x_px = (x_m / Width_m) * Width_px
    #   y_px = (H_px - (y_m / Height_m) * H_px)  <-- FLIP Y axis if needed (standard image coords)
    #   (Check if your map origin (0,0) is Top-Left or Bottom-Left. Usually maps are Bottom-Left, images Top-Left)
    
    pixels = np.zeros((N, 2))
    pixels[:, 0] = (points_data[:, 0] / width_m) * W_pixels   # x
    pixels[:, 1] = H_pixels - ((points_data[:, 1] / height_m) * H_pixels) # y (flipped)
    
    # Clip to be safe inside image bounds
    pixels[:, 0] = np.clip(pixels[:, 0], 0, W_pixels - 1)
    pixels[:, 1] = np.clip(pixels[:, 1], 0, H_pixels - 1)

    print(f"Building graph for {N} nodes. Checking mask constraints...")
    count_edges = 0
    
    # --- 4. Connect Nodes ---
    for i in range(N):
#         px1, py1 = pixels[i]
#         x1, y1 = points_data[i]
        
#         for j in range(i+1, N):
#             px2, py2 = pixels[j]
#             x2, y2 = points_data[j]
            
#             # A. Distance Check (Euclidean in Meters)
#             dist = float(np.hypot(x2 - x1, y2 - y1))
            
#             # OPTIONAL BUT RECOMMENDED: Max Connection Radius
#             # If nodes are > 20m apart, don't connect them. Improves sparsity and structure.
#             if dist > 30.0: 
#                 continue

#             # B. Wall Check (in Pixels)
#             if obey_mask:
#                 # segment_inside_mask should check if the line (px1, py1) -> (px2, py2) hits a wall
#                 if not segment_inside_mask(mask, (px1, py1), (px2, py2), thickness=thickness):
#                     continue

#             G.add_edge(i, j, weight=dist)
#             count_edges += 1
            
#     print(f"✅ created graph: {G.number_of_nodes()} nodes, {count_edges} edges")
#     return G

# def create_complete_graph_from_points(mask, points, width_m, height_m, obey_mask=True, thickness=2):
#     # ... (Keep the setup code: Standardizing input, H_pixels, W_pixels calculation) ...
#     # Ensure you have the 'pixels' array calculated correctly as before
    
#     # ... (Inside the loop) ...
#     for i in range(N):
        px1, py1 = pixels[i]
        x1, y1 = points_data[i]
        
        for j in range(i+1, N):
            px2, py2 = pixels[j]
            x2, y2 = points_data[j]
            
            # 1. Base Euclidean Distance
            dist = float(np.hypot(x2 - x1, y2 - y1))
            
            # 2. Connection Radius (e.g., 30m)
            if dist > 30.0: 
                continue

            # 3. SOFT WALL LOGIC
            weight = dist
            
            if obey_mask:
                # Check if line hits a wall
                is_clear = segment_inside_mask(mask, (px1, py1), (px2, py2), thickness=thickness)
                
                if not is_clear:
                    # INSTEAD OF "continue" (Deleting), we PENALIZE
                    # Walls attenuate signals. Let's say a wall makes the distance feel 2.5x longer.
                    WALL_PENALTY = 2.5 
                    weight = dist * WALL_PENALTY
            
            # Add the edge with the (potentially penalized) weight
            G.add_edge(i, j, weight=weight)
            count_edges += 1
            
    print(f"✅ Created Soft-Wall Graph: {G.number_of_nodes()} nodes, {count_edges} edges")
    return G

# def create_complete_graph_from_points(mask, points, obey_mask=True, thickness=2):
#     """
#     points: np.ndarray (N, 2) in (x,y) order or DataFrame
#     mask: 2D uint8, 0/255
#     returns: networkx.Graph with all nodes and edges
#     """
#     # --- 1. Standardization Step ---
#     if isinstance(points, pd.DataFrame):
#         # Check for common column names. 
#         # Note: The original docstring said "in (y, x) order". 
#         # If your DF has 'x' and 'y' columns, we explicitly map them.
#         if 'x' in points.columns and 'y' in points.columns:
#             # We extract them in (y, x) order to match the original function's logic
#             # where: for i, (y, x) in enumerate(points)
#             points_data = points[['x', 'y']].values
#         else:
#             # If no labels, just convert to numpy array assuming (y, x) structure
#             points_data = points.values
#     else:
#         # It's already an array or list
#         points_data = points
        
#     G = nx.Graph()
#     N = len(points)
#     if N == 0:
#         print("⚠️ no points given")
#         return G

#     # 1) add nodes with pos
#     for i, (x,y) in enumerate(points_data):
#         G.add_node(i, pos=(float(x), float(y)))

#     # 2) connect everyone to everyone
#     for i in range(N):
#         x1, y1 = G.nodes[i]["pos"]
#         for j in range(i+1, N):
#             x2, y2 = G.nodes[j]["pos"]

#             if obey_mask:
#                 if not segment_inside_mask(mask, (x1, y1), (x2, y2), thickness=thickness):
#                     continue

#             dist = float(np.hypot(x2 - x1, y2 - y1))
#             G.add_edge(i, j, weight=dist)

#     print(f"✅ created graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
#     return G

def build_graph_from_points(mask, points, radius=None, k=6):
    """
    Build a graph on sampled points.
    - If radius is given -> connect pairs within that distance.
    - Else -> connect each point to its k nearest neighbors.
    Edge weight = Euclidean distance.
    """
    G = nx.Graph()
    if len(points) == 0:
        return G

    # add nodes with positions (x,y) for plotting
    for i, (y, x) in enumerate(points):
        G.add_node(i, pos=(int(x), int(y)))

    tree = cKDTree(points)

    if radius is not None:
        # radius graph
        for i, j in tree.query_pairs(r=radius):
            (y1, x1), (y2, x2) = points[i], points[j]
            if mask[int(y1), int(x1)] > 0 and mask[int(y2), int(x2)] > 0:
                w = float(np.hypot(x2 - x1, y2 - y1))
                G.add_edge(i, j, weight=w)
    else:
        # k-NN graph (k>=1). query k+1 because the nearest neighbor is the point itself
        k = max(1, int(k))
        dists, idxs = tree.query(points, k=k+1)
        for i, (row_d, row_idx) in enumerate(zip(dists, idxs)):
            for d, j in zip(row_d[1:], row_idx[1:]):  # skip self
                (y1, x1), (y2, x2) = points[i], points[j]
                if mask[int(y1), int(x1)] > 0 and mask[int(y2), int(x2)] > 0:
                    G.add_edge(i, j, weight=float(d))
    
    

    return G


def compute_shortest_paths(G, points, circles):
    distance_matrix = np.zeros((len(points), len(circles)))
    for i, (px, py) in enumerate(points):
        if (i==10):
            print(i)
        for j, (cx, cy, _) in enumerate(circles):
            try:
                path_length = nx.shortest_path_length(G, (py, px), (cy, cx))
            except nx.NetworkXNoPath:
                path_length = float('inf')
            distance_matrix[i, j] = path_length
    return distance_matrix

def plot_original_image(file_path, site, floor):
    img_path = create_img_path(file_path, site, floor)
    img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    
    # handle transparency
    if img is None:
        raise ValueError(f"Could not read {img_path}")
    
    if img.shape[2] == 4:
        img_rgb = cv2.cvtColor(img[:, :, :3], cv2.COLOR_BGR2RGB)
    else:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10, 10))
    plt.imshow(img_rgb)
    plt.title("Original Floorplan Image")
    plt.axis("off")
    plt.show()

def create_img_path(files_dir_path, site,floor):
    fstr = f"F{floor}" if not str(floor).upper().startswith("F") else str(floor).upper()
    img_path = next((Path(files_dir_path)/site/fstr).glob("floor_image.*"))
    return img_path

In [6]:
# ---------- 1) INSIDE via alpha outer contour ----------
def inside_mask_from_alpha_outer_contour(img_path, alpha_clear_thr=10, close_ks=5):
    img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f"Could not read {img_path}")
    if img.ndim < 3 or img.shape[2] < 4:
        raise ValueError("PNG with alpha is required.")

    H, W = img.shape[:2]
    alpha = img[:, :, 3]
    opaque = (alpha > alpha_clear_thr).astype(np.uint8) * 255

    if close_ks > 0:
        k = cv2.getStructuringElement(cv2.MORPH_RECT, (close_ks, close_ks))
        opaque = cv2.morphologyEx(opaque, cv2.MORPH_CLOSE, k)

    cnts, _ = cv2.findContours(opaque, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return np.zeros((H, W), np.uint8)

    mall_cnt = max(cnts, key=cv2.contourArea)
    inside = np.zeros((H, W), np.uint8)
    cv2.drawContours(inside, [mall_cnt], -1, 255, thickness=cv2.FILLED)
    return inside

# ---------- 2) “Store” detection (your gates) ----------
def detect_store_mask(
    img_bgr,
    inside_mask,
    *,
    black_V_max=110, close_iters=3, dilate_black_px=1,
    white_S_max=80,  white_V_min=185,
    pastel_S_max=90, pastel_V_min=165,
    cyan_H=(80,100),  cyan_S_min=25,  cyan_V_min=110,
    blue_H=(90,140),  blue_S_min=25,  blue_V_min=110,
    purple_H=(120,175), purple_S_min=20, purple_V_min=105,
    min_area_px=80, shrink_px=1
):
    """Return mask (255) for store polygons inside the black frame."""
    H, W = img_bgr.shape[:2]
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # black outline/walls
    black = cv2.inRange(hsv, np.array([0,0,0], np.uint8),
                             np.array([180,255,black_V_max], np.uint8))
    black = cv2.morphologyEx(black, cv2.MORPH_CLOSE, np.ones((3,3), np.uint8), iterations=close_iters)
    if dilate_black_px > 0:
        k = 2*dilate_black_px+1
        black = cv2.dilate(black, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k,k)), 1)

    # broad light/white
    white = cv2.inRange(hsv, np.array([0,0,white_V_min], np.uint8),
                             np.array([180,white_S_max,255], np.uint8))

    # pastel gate: low S, high V (any hue)
    s = hsv[...,1]; v = hsv[...,2]
    pastel = np.zeros((H,W), np.uint8)
    pastel[(s <= pastel_S_max) & (v >= pastel_V_min)] = 255

    # explicit hues
    def hue_mask(h_lo, h_hi, s_min, v_min):
        return cv2.inRange(hsv,
                           np.array([h_lo, s_min, v_min], np.uint8),
                           np.array([h_hi, 255, 255], np.uint8))
    cyan   = hue_mask(*cyan_H,   cyan_S_min,   cyan_V_min)
    blue   = hue_mask(*blue_H,   blue_S_min,   blue_V_min)
    purple = hue_mask(*purple_H, purple_S_min, purple_V_min)

    # union, constrained to inside and not on the black walls
    stores = white | pastel | cyan | blue | purple
    stores = cv2.bitwise_and(stores, inside_mask)
    stores = cv2.bitwise_and(stores, cv2.bitwise_not(black))

    # clean + optional shrink so we don't touch wall pixels
    stores = cv2.morphologyEx(stores, cv2.MORPH_OPEN, np.ones((3,3), np.uint8), 1)

    if shrink_px > 0:
        k = 2*shrink_px+1
        stores = cv2.erode(stores, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k,k)), 1)

    # area filter
    cnts, _ = cv2.findContours(stores, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    keep = np.zeros_like(stores)
    for c in cnts:
        if cv2.contourArea(c) >= min_area_px:
            cv2.drawContours(keep, [c], -1, 255, thickness=cv2.FILLED)
    return keep

# ---------- 3) Final: inside minus stores = corridors/voids ----------
def get_corridor_mask(img_path,
                      alpha_clear_thr=10, close_ks=5,
                      **store_gate_kwargs):
    img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f"Could not read {img_path}")
    bgr = img[:, :, :3] if (img.ndim==3 and img.shape[2]==4) else img

    inside = inside_mask_from_alpha_outer_contour(img_path, alpha_clear_thr, close_ks)
    store_mask = detect_store_mask(bgr, inside, **store_gate_kwargs)
    corridor = cv2.bitwise_and(inside, cv2.bitwise_not(store_mask))
    return corridor, store_mask, inside

# ---------- Overlay helper ----------
def overlay_mask_yellow(img_path, mask, alpha=0.6, out_path=None):
    img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    base = img[:, :, :3] if (img.ndim==3 and img.shape[2]==4) else img
    overlay = base.copy()
    overlay[mask>0] = (0,255,255)  # yellow (BGR)
    out = cv2.addWeighted(overlay, alpha, base, 1-alpha, 0)
    if out_path: cv2.imwrite(out_path, out)
    return out

def thin_black_text_mask(img_bgr, inside_mask=None,
                         v_thr=105, s_thr=110,  # what counts as "black"
                         min_stroke_px=4):      # remove strokes thinner than ~this width
    """
    Returns 255 where black strokes are *thin* (likely text).
    min_stroke_px is the approximate full stroke width in pixels.
    """
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    hsv  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    V, S = hsv[:, :, 2], hsv[:, :, 1]

    # Black pixels (very dark & not saturated)
    black = ((V < v_thr) & (S < s_thr)).astype(np.uint8) * 255

    # Optional: only care about thin black *inside* the mall
    if inside_mask is not None:
        black = cv2.bitwise_and(black, inside_mask)

    # Distance transform on black foreground
    # (distance ~ half the local stroke width)
    dist = cv2.distanceTransform(black, cv2.DIST_L2, 3)

    # Pixels with half-width < min_stroke_px/2 are thin
    thin = (dist < (min_stroke_px / 2.0)).astype(np.uint8) * 255

    # Clean specks
    thin = cv2.morphologyEx(thin, cv2.MORPH_OPEN, np.ones((3,3), np.uint8), 1)
    return thin

def corridors_without_text(img_path,get_inside_fn,      # your inside-mask function
                           detect_stores_fn,   # your store mask function (gates)
                           **kwargs_for_stores):
    img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    bgr = img[:, :, :3] if (img.ndim==3 and img.shape[2]==4) else img

    # 1) inside
    inside = get_inside_fn(img_path)

    # 2) stores
    stores = detect_stores_fn(bgr, inside, **kwargs_for_stores)

    # 3) corridors (inside minus stores)
    corridor = cv2.bitwise_and(inside, cv2.bitwise_not(stores))

    # 4) remove thin black strokes (text)
    thin_text = thin_black_text_mask(bgr, inside_mask=inside,
                                     v_thr=105, s_thr=110,   # tune if needed
                                     min_stroke_px=4)        # ~4–6 works well
    corridor = cv2.bitwise_and(corridor, cv2.bitwise_not(thin_text))
    return corridor

# Use your store detector (the v3 gates you posted):
def stores_fn(img_bgr, inside_mask, **k):
    return detect_store_mask(
        img_bgr, inside_mask,
        black_V_max=110, close_iters=3, dilate_black_px=1,
        white_S_max=80, white_V_min=185,
        pastel_S_max=90, pastel_V_min=165,
        cyan_H=(80,100),  cyan_S_min=25,  cyan_V_min=110,
        blue_H=(90,140),  blue_S_min=25,  blue_V_min=110,
        purple_H=(120,175), purple_S_min=20, purple_V_min=105,
        min_area_px=80, shrink_px=1
    )

def sample_points_from_mask(mask, n=100, value_threshold=128, seed=None):
    """
    Randomly sample N valid (y,x) points from a binary mask.

    Parameters
    ----------
    mask : np.ndarray (H x W)
        Binary or grayscale mask. Non-zero = valid region.
    n : int
        Number of points to sample.
    value_threshold : int
        Minimum pixel value to count as 'inside' (default 128).
    seed : int, optional
        Random seed for reproducibility.

    Returns
    -------
    pts : np.ndarray of shape (n, 2)
        Randomly sampled points in (y, x) order.
    """
    if seed is not None:
        np.random.seed(seed)

    # Find all valid coordinates
    ys, xs = np.where(mask > value_threshold)
    if len(ys) == 0:
        raise ValueError("Mask has no valid pixels above threshold.")

    # If there are fewer valid pixels than n, sample with replacement
    idx = np.random.choice(len(ys), size=n, replace=(len(ys) < n))
    pts = np.column_stack((ys[idx], xs[idx]))
    return pts


# Robust example

In [7]:
class PhysicalLayoutGraph:
    def __init__(self,floor, site,files_dir_path,  num_points=200, embedding_dim=EMBEDDING_DIM, k=4):
        self.floor = floor
        self.site = site
        self.image_path = create_img_path(files_dir_path, site, floor)

        self.num_points = num_points
        self.embedding_dim = embedding_dim
        self.k = k
        
        self.random_points = None
        self.L = None
        floor_info_path = Path(files_dir_path)/site/f"F{floor}" / "floor_info.json"
        
        with open(floor_info_path) as f:
            inf = json.load(f)
        self.width_meter, self.height_meter = inf["map_info"]["width"], inf["map_info"]["height"]

        # self._process_image()
        self.create_corridors_mask()
        self.sample_points_from_mask()
        # self._center_layout()

        # self._build_graph()
        # self.plot_layout_graph()
        # self._compute_adjacency()
        self.embedding = None
        self.dist_matrix = None
    
    def save_points(self,points):
        self.random_points = points
        
    def sample_points_from_mask(self):
        self.random_points = sample_points_from_mask(self.polygon_mask, n=self.num_points)
   
    def _process_image(self):
        image, gray = load_image(self.image_path)
        self.image = image
        self.gray = gray
        self.polygon_mask, self.rectangles, self.circles = detect_shapes(gray)
        #self.df_points_with_signals = {'x': random_points_from_graph[:, 0], 'y': random_points_from_graph[:, 1]}

    def create_corridors_mask(self):
        # Use the inside-mask function you liked ("outside_final" logic → inside):
        inside_fn = lambda p: inside_mask_from_alpha_outer_contour(p, alpha_clear_thr=10, close_ks=7)
        self.polygon_mask = corridors_without_text(self.image_path, inside_fn, stores_fn)
        self.circles = None
        self.rectangles = None

    def _center_layout(self):
        """Center only the random points."""
        h, w = self.polygon_mask.shape
        self.center_point = np.array([w // 2, h // 2])
        self.random_points = np.array(generate_random_points(self.polygon_mask, num_points=self.num_points))
        # self.A_mean = np.round(A_all.mean(axis=0, keepdims=True)).astype(int)
        # Center only the random points
        # self.random_points = A_all - self.A_mean
        
    # def _build_graph(self):
    #      # self.polygon_mask  -> 2D mask
    #     # self.random_points -> (N,2) (y,x)
    #     self.G = create_complete_graph_from_points(
    #         self.polygon_mask,
    #         self.random_points,
    #         obey_mask=True,   # set to False if you want truly complete
    #         thickness=2
    #     )

    #     print("nodes:", self.G.number_of_nodes())
    #     print("edges:", self.G.number_of_edges())
    #     print("first 3 nodes:", list(self.G.nodes(data=True))[:3]) 
    #     # self.G = build_graph_from_points(self.polygon_mask, self.random_points,  k=2)
    #     # removed_edges = remove_edges_outside_mask(self.G, self.polygon_mask)
    #     # self.random_points = generate_random_points(self.polygon_mask, num_points=self.num_points)
    #     # A_all = np.array(self.random_points)
    #     # self.A_mean = np.round(A_all.mean(axis=0, keepdims=True)).astype(int)
    #     # centered_A = A_all - self.A_mean
    #     # self.random_points = centered_A

    # In your PhysicalLayoutGraph class:

    def _build_graph(self):
        # Ensure you have width and height defined. 
        # If not, define them here or pass them in.
        # Example values: width_meter=150.0, height_meter=80.0 (Replace with YOUR real values)
        
        self.G = create_complete_graph_from_points(
            mask=self.polygon_mask,
            points=self.random_points,
            width_m=self.width_meter,   
            height_m=self.height_meter,  
            obey_mask=True,
            thickness=2
        )
        # 3. CRITICAL FIX: Remove Islands immediately
        # If we still have small islands, delete them so they don't break the embedding
        # if not nx.is_connected(self.G):
        #     print(f"⚠️ Graph has {nx.number_connected_components(self.G)} components. Keeping only the largest.")
        #     largest_cc = max(nx.connected_components(self.G), key=len)
            
        #     # Update random_points to keep only the connected ones
        #     valid_indices = list(largest_cc)
        #     self.random_points = self.random_points.loc[valid_indices]
            
        #     # Rebuild graph one last time with only valid points (clean)
        #     self.G = self.G.subgraph(largest_cc).copy()
        #     # Relabel nodes to be 0..N-1
        #     self.G = nx.convert_node_labels_to_integers(self.G)
            
        # print(f"✅ Final Graph: {self.G.number_of_nodes()} nodes, {self.G.number_of_edges()} edges. Connected? {nx.is_connected(self.G)}")

    def plot_layout_graph(self):
        """
        Plots the layout graph with nodes, edges, and optional circles over the polygon mask.

        Parameters
        ----------
        G : networkx.Graph
            Graph with 'pos' attributes for nodes.
        polygon_mask : np.ndarray
            Background grayscale mask (e.g., corridor mask or layout mask).
        highlighted_nodes : list[int], optional
            Nodes to highlight in a different color.
        circles : list[(x, y, r)], optional
            Circles to draw (if available).
        """
        pos = nx.get_node_attributes(self.G, 'pos')

        num_nodes = self.G.number_of_nodes()
        num_edges = self.G.number_of_edges()
        print(f"🧩 Plotting layout: {num_nodes} nodes, {num_edges} edges")
        if num_edges == 0:
            print("⚠️ No edges to plot. Possibly all were removed.")
            return None, None
    
        # Node colors
        node_colors = ["green"] * num_nodes

        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(self.polygon_mask, cmap='gray', alpha=0.3)

        # Draw edges and nodes
        nx.draw(
            self.G, pos, with_labels=False,
            node_size=40, node_color=node_colors,
            edge_color='blue', alpha=0.8, ax=ax
        )

        # Draw optional circles (if provided)
        if self.circles:
            pass
            # for (x, y, r) in self.circles:
            #     circ = plt.Circle((x, y), r, color="red", fill=False, lw=1)
            #     ax.add_patch(circ)

        ax.set_title(f"Graph made with sampled points from the walkable area.\n site:{self.site}\n floor:{self.floor}", fontsize=14)
        ax.axis("off")
        plt.tight_layout()
        return fig, ax

    def _compute_adjacency(self):
        self.kernel_matrix = compute_kernel_matrix(self.random_points, k=self.k)

    def _compute_embedding(self, embedding_dim = EMBEDDING_DIM):
        self.L = compute_laplacian(self.kernel_matrix)
        print(embedding_dim + 1)
        lam, v = compute_leading_eigenvectors(self.L, embedding_dim + 1)
        self.embedding = v[:,  1:embedding_dim + 1]

    def get_embedding(self):
        return self.embedding

    def get_points(self):
        return self.random_points

    def plot_embedding(self):
        plt.figure(figsize=(6, 6))
        plt.scatter(self.embedding[:, 0], self.embedding[:, 1])
        plt.title("Physical Layout Spectral Embedding")
        plt.show()

    def plot_on_layout(self, predicted_points, true_points, title="Predicted Layout Locations"):
      plt.figure(figsize=(8, 8))
      plt.imshow(self.polygon_mask, cmap='gray', alpha=0.5)
      plt.imshow(self.image, alpha=0.3)

      predicted_p = np.array(predicted_points)
      plt.scatter(predicted_p[:, 0], predicted_p[:, 1], color='red', s=50, label="Predicted Points")

      true_p = np.array(true_points)
      plt.scatter(true_p[:, 0], true_p[:, 1], color='green', s=50, label="True Points")

      plt.legend()
      plt.title(title)
      plt.show()

    def embedding_to_layout_xy(self, embedding_points):
        """
        Maps points from embedding space to nearest real (x,y) layout points.
        """
        # Build KDTree of embedding
        tree = cKDTree(self.embedding)

        # Find nearest neighbor indices
        dists, indices = tree.query(embedding_points)

        # Map to (X, Y) points
        real_points = np.array(self.random_points)[indices]

        return real_points



In [8]:
class SignalGraph:
    def __init__(self, signal_data, embedding_dim=EMBEDDING_DIM, k_nearest=1):
        self.signal_data = signal_data
        self.embedding_dim = embedding_dim
        self.k_nearest = k_nearest
        self.embedding = None
        self.L= None
        self._build_kernel()
        
    def _build_kernel(self):
        self.kernel_matrix = compute_kernel_matrix(self.signal_data, k=self.k_nearest)
        # self.kernel_matrix = compute_robust_kernel(self.signal_data, k=20)

    def _compute_embedding(self, embedding_dim):
        self.L = compute_laplacian(self.kernel_matrix)
        lam, v = compute_leading_eigenvectors(self.L, embedding_dim + 1)
        self.embedding = v[:, 1:embedding_dim + 1]

    def get_embedding(self):
        return self.embedding

    def plot_embedding(self):
        plt.figure(figsize=(6, 6))
        plt.scatter(self.embedding[:, 0], self.embedding[:, 1])
        plt.title("Signal Graph Spectral Embedding")
        plt.show()


    def compute_robust_kernel(fingerprints, k=10):
        # 1. Use COSINE distance (1 - cosine_similarity)
        # robust to absolute signal strength differences
        dists = pdist(fingerprints, metric='cosine')
        D = squareform(dists)
        
        # 2. Adaptive Sigma (Self-Tuning)
        # sigma_i = distance to k-th neighbor
        sorted_dists = np.sort(D, axis=1)
        sigma = sorted_dists[:, k]  # (N,)
        
        # 3. Construct Kernel
        N = D.shape[0]
        K = np.zeros((N, N))
        
        # Vectorized calculation for speed
        # K_ij = exp( - dist^2 / (sigma_i * sigma_j) )
        # We use outer product to get matrix of sigma_i * sigma_j
        sig_prod = np.outer(sigma, sigma)
        
        # Avoid division by zero
        sig_prod[sig_prod < 1e-8] = 1.0 
        
        K = np.exp(- (D**2) / sig_prod)
        
        return K



In [9]:
class ManifoldMatcherSSL:
    def __init__(self):
        self.fitted = False
        self.layout_graph = None
        self.signal_graph = None
        self.aligned_signal_embedding = None
        self.labeled_indices = None 

    def fit(self, layout_graph, signal_graph, labeled_indices):
        """
        Calibrate layout to signal space using SSL-style least squares.
        """
        self.layout_graph = layout_graph
        self.signal_graph = signal_graph

        # phi_A = layout_embedding[labeled_indices]
        phi_S = self.signal_graph.embedding[labeled_indices]

        #centered
        A_all = self.layout_graph.random_points[['x', 'y']].values
        self.A_mean = A_all.mean(axis=0, keepdims=True)  
        A_labeled = layout_graph.random_points.loc[labeled_indices, ['x', 'y']].values
        
        centered_A = A_labeled - self.A_mean

        self.C, _, _, _ = np.linalg.lstsq( phi_S, centered_A, rcond=None)
        print(f"phi_S_dim :{phi_S.shape}, A_dim:{centered_A.shape}, C dim {self.C.shape}, ")
        print(self.C)
        A_all_centered = A_all - self.A_mean
        self.calibrated_phi_s = phi_S  @ self.C
        self.fitted = True
        return phi_S, centered_A, self.C, self.calibrated_phi_s

    def predict(self, validation_indices):
        """
        Predict layout locations for new signal vectors (not embeddings).
        Uses 1-NN matching in calibrated spectral space.
        """
        if not self.fitted:
            raise ValueError("Must call .fit() before .predict()")
        self.calibrated_phi_s = self.signal_graph.embedding[validation_indices] @ self.C
        return self.calibrated_phi_s + self.A_mean
 

In [ ]:
class ManifoldMatcher:
    def __init__(self, layout_graph= None, signal_graph = None):
        self.fitted = False
        self.layout_graph = layout_graph
        self.signal_graph = signal_graph
        self.aligned_signal_embedding = None
        self.labeled_indices = None 

    def fit(self, labeled_indices,num_eig_vec):
        """
        Calibrate layout to signal space using SSL-style least squares.
        """
        # phi_S = self.signal_graph.embedding[labeled_indices]
        # phi_A = self.layout_graph.embedding[labeled_indices]

        layout_positions = self.layout_graph.random_points.index.get_indexer(labeled_indices)
    
        # בדיקה: אם יש אינדקס שלא נמצא (יחזיר 1-), נסנן אותו כדי למנוע קריסה
        mask = layout_positions != -1
        print(mask)
        valid_layout_pos = layout_positions[mask]
        valid_labeled_indices = np.array(labeled_indices)[mask]

        # 2. עכשיו השליפה תעבוד מושלם כי אנחנו משתמשים במיקומים הפיזיים (Positions)
        phi_A = self.layout_graph.embedding[valid_layout_pos, :num_eig_vec]
        
        # עבור הסיגנלים - אם הם לא עברו סינון, אפשר להשתמש ב-indices המקוריים
        # אבל ליתר ביטחון, נשתמש רק באלו שמצאנו להם התאמה ב-Layout
        phi_S = self.signal_graph.embedding[valid_labeled_indices, :num_eig_vec]

        print(f"phi_S_dim :{phi_S.shape},phi_A_dim:{phi_A.shape}, ")

        self.C, _, _, _ = np.linalg.lstsq( phi_S, phi_A, rcond=None)
        self.calibrated_phi_s_labeled = phi_S  @ self.C
        self.fitted = True
        return phi_S, self.C, self.calibrated_phi_s_labeled 

    def predict(self, validation_signal_data,num_eig_vec, normalize_flag=False):
        """
        Predict layout locations for new signal vectors (not embeddings).
        Uses 1-NN matching in calibrated spectral space.
        """
        if not self.fitted:
            raise ValueError("Must call .fit() before .predict()")
        # self.calibrated_phi_s = self.calibrated_phi_s + self.A_mean
        # print(self.signal_graph.embedding[validation_indices].shape)
        # print(self.C.shape)
        # self.calibrated_phi_s = self.signal_graph.embedding[validation_indices,1:3] @ self.C[1:3,:]
        # val_pos = val_pos[val_pos != -1]
        # print(validation_signal_data.index)
        validation_indices = (validation_signal_data.index).to_list()
        # print(validation_indices)
        val_pos = self.signal_graph.signal_data.index.get_indexer(validation_indices)
        # print(val_pos) # מסנן אינדקסים שלא נמצאו
        self.calibrated_phi_s = self.signal_graph.embedding[validation_indices,0:num_eig_vec] @ self.C[0:num_eig_vec,:]
        
        # [v1_phis_min,v2_phis_min,v3_phis_min, v4] =  np.min(self.calibrated_phi_s, axis=0)
        # [v1_phis_max,v2_phis_max,v3_phis_max, v4max] =  np.max(self.calibrated_phi_s, axis=0)

        # [v1_lay_emb_min,v2_lay_emb_min,v3_lay_emb_min, v44] =  np.min(self.layout_graph.embedding, axis=0)
        # [v1_lay_emb_max,v2_lay_emb_max,v3_lay_emb_max, v44max] =  np.max(self.layout_graph.embedding, axis=0)
        # print("-----min---------")
        # print(v1_phis_min, v1_lay_emb_min)
        # print(v2_phis_min, v2_lay_emb_min)
        # print(v3_phis_min, v3_lay_emb_min)
        # print("-------max-------")
        # print(v1_phis_max, v1_lay_emb_max)
        # print(v2_phis_max, v2_lay_emb_max)
        # print(v3_phis_max, v3_lay_emb_max)

        if normalize_flag:
            norm_calibrated_signal = normalize(self.calibrated_phi_s, axis=1)
            norm_layout_embedding = normalize(self.layout_graph.embedding, axis=1)
            D = cdist(norm_calibrated_signal, norm_layout_embedding, metric='euclidean')
        else:
        # No need to normalize because the n_points in layout graph = n_points in signal_graph
            D = cdist(self.calibrated_phi_s, self.layout_graph.embedding, metric='euclidean')
        min_idx = np.argmin(D, axis=1)
        
        return self.layout_graph.random_points.iloc[min_idx]
    
    def predict2(self, signal_indices_to_predict, num_eig_vec):
        # 1. חישוב ה-Embedding המיושר הנוכחי
        predict_sig_pos = self.signal_graph.signal_data.index.get_indexer(signal_indices_to_predict)
        S_aligned = self.signal_graph.embedding[predict_sig_pos, :num_eig_vec] @ self.C[0:num_eig_vec,:] 
        
        # 2. מתיחת הטווחים (Per-Vector Scaling)
        # אנחנו מותחים כל וקטור אדום שייגע בקצוות של הוקטור האפור המקביל
        self.S_stretched = S_aligned.copy()
        for i in range(S_aligned.shape[1]):
            # טווח המטרה (Layout)
            l_min, l_max = self.layout_graph.embedding[:, i].min(), self.layout_graph.embedding[:, i].max()
            # טווח נוכחי (Signals)
            s_min, s_max = S_aligned[:, i].min(), S_aligned[:, i].max()
            
            # נוסחת המתיחה
            self.S_stretched[:, i] = l_min + (S_aligned[:, i] - s_min) * (l_max - l_min) / (s_max - s_min)

        # 3. חישוב מרחקים על המטריצה המתוחה
        D = cdist(self.S_stretched, self.layout_graph.embedding, metric='euclidean')
        min_idx = np.argmin(D, axis=1)
        
        return self.layout_graph.random_points.iloc[min_idx]

    def predict_var_match(self, validation_indices,num_eig_vec, num_neighbors=1):
        # 1. הכנת הנתונים (כמו בקוד המקורי שלך)
        # שימוש ב-get_indexer לתרגום אינדקסים למנוע קריסות
        val_pos = self.signal_graph.signal_data.index.get_indexer(validation_indices)
        mask = val_pos != -1
        valid_pos = val_pos[mask]
        
        # שליפת הסיגנלים
        phi_S = self.signal_graph.embedding[valid_pos]
        
        # 2. הטרנספורמציה המקורית (שכבר עובדת לך מבחינת כיוון)
        # self.C נלמד ב-fit בעזרת lstsq
        predicted_embedding = phi_S[:, :num_eig_vec] @ self.C[0:num_eig_vec,:]
        
        # --- התיקון מתחיל כאן ---
        # 3. Variance Matching (התאמת אנרגיה)
        # אנחנו בודקים: מה "רוחב" הפיזור של הלייאאוט ומה "רוחב" הפיזור של התחזית שלנו?
        
        # חישוב סטיית התקן של כל וקטור בלייאאוט (המטרה שלנו)
        std_layout = np.std(self.layout_graph.embedding, axis=0)
        
        # חישוב סטיית התקן של התחזית הנוכחית (שיצאה "מכווצת")
        std_pred = np.std(predicted_embedding, axis=0)
        
        # חישוב פקטור התיקון: כמה צריך "למתוח" כל וקטור כדי שיחזור לגודל המקורי?
        # מוסיפים epsilon קטן למנוע חילוק באפס
        scaling_factor = std_layout / (std_pred + 1e-8)
        
        # יישום המתיחה: זה "זורק" את הנקודות מהמרכז החוצה לקצוות
        self.predicted_embedding_scaled = predicted_embedding * scaling_factor
        # --- סוף התיקון ---

        # 4. מציאת השכן הקרוב (Nearest Neighbor) על המרחב המתוקן
        dists = cdist(self.predicted_embedding_scaled, self.layout_graph.embedding, metric='euclidean')
        min_idx = np.argmin(dists, axis=1)
        
        # החזרת התוצאות
        return self.layout_graph.random_points.iloc[min_idx]
    
    def predict_hist_match(self, validation_indices, num_eig_vec, k_neighbors=1):
        # 1. הכנת נתונים ושליפה
        # תרגום אינדקסים למיקומים בזיכרון
        val_pos = self.signal_graph.signal_data.index.get_indexer(validation_indices)
        
        # סינון נקודות לא קיימות (אם יש)
        mask = val_pos != -1
        valid_pos = val_pos[mask]
        
        # שליפת הסיגנלים
        phi_S = self.signal_graph.embedding[valid_pos]
        print(len(phi_S))
        # 2. הטרנספורמציה הלינארית עם num_eig_vec
        # כאן אנחנו משתמשים רק בכמות הוקטורים שהגדרת כדי לחזות את המיקום
        # (מניח ש-self.C נלמד במימדים המתאימים או שהוא מכיל את כולם)
        predicted_embedding = phi_S[:, :num_eig_vec] @ self.C[0:num_eig_vec, :]
        
        # --- התיקון: Histogram Matching ---
        # אנחנו מכריחים את ההתפלגות של התחזית להיות זהה להתפלגות של הלייאאוט
        matched_embedding = np.zeros_like(predicted_embedding)
        
        # לולאה על כל מימד (וקטור עצמי) בנפרד
        for i in range(predicted_embedding.shape[1]):
            # א. נתוני המטרה (הלייאאוט) ממויינים
            # הערה: אם num_eig_vec קטן ממספר העמודות ב-C, אנחנו עדיין מתאימים לכל עמודות הפלט שנוצרו
            layout_vals = np.sort(self.layout_graph.embedding[:, i])
            
            # ב. נתוני התחזית שלנו
            pred_vals = predicted_embedding[:, i]
            
            # ג. חישוב האחוזונים (Quantiles) של התחזית
            from scipy.stats import rankdata
            quantiles = (rankdata(pred_vals) - 1) / (len(pred_vals) - 1)
            
            # ד. מיפוי הערכים מהלייאאוט לפי האחוזונים
            matched_embedding[:, i] = np.interp(quantiles, np.linspace(0, 1, len(layout_vals)), layout_vals)
        
        # 3. מציאת השכנים הקרובים (K-NN) על המרחב המתוקן
        self.matched_embedding = matched_embedding
        # אנחנו משווים את ה-Embedding המתוקן ל-Embedding המקורי של הלייאאוט
        dists = cdist(matched_embedding, self.layout_graph.embedding, metric='euclidean')
        
        if k_neighbors > 1:
            # ממוצע של K השכנים הקרובים (מחליק רעשים)
            closest_indices = np.argsort(dists, axis=1)[:, :k_neighbors]
            
            # שליפת הקואורדינטות הפיזיות (x, y) לכל שכן
            all_coords = np.array([self.layout_graph.random_points.iloc[idx][['x', 'y']].values 
                                for idx in closest_indices])
            
            # חישוב הממוצע
            avg_coords = all_coords.mean(axis=1)
            
            # החזרה כ-DataFrame
            res = pd.DataFrame(avg_coords, columns=['x', 'y'], index=validation_indices)
            return self.layout_graph.random_points.iloc[closest_indices]
            
        else:
            # השכן הקרוב ביותר בלבד (גרסה פשוטה)
            min_idx = np.argmin(dists, axis=1)
            
            res = self.layout_graph.random_points.iloc[min_idx].copy()
            # וידוא שהאינדקסים חוזרים להיות השמות המקוריים של נקודות הוידוא
            res.index = validation_indices 
            return res
        
    def predict_weighted(self, validation_indices, num_eig_vec=3, k_neighbors=1):
        # 1. הכנת נתונים
        val_pos = self.signal_graph.signal_data.index.get_indexer(validation_indices)
        mask = val_pos != -1
        valid_pos = val_pos[mask]
        phi_S = self.signal_graph.embedding[valid_pos]
        
        # 2. חיזוי ראשוני (הטרנספורמציה הליניארית)
        predicted_embedding = phi_S[:, :num_eig_vec] @ self.C[0:num_eig_vec, :]
        
        # 3. Histogram Matching (חיוני כדי למנוע את הגושים שראית)
        matched_embedding = np.zeros_like(predicted_embedding)
        for i in range(predicted_embedding.shape[1]):
            layout_vals = np.sort(self.layout_graph.embedding[:, i])
            pred_vals = predicted_embedding[:, i]
            from scipy.stats import rankdata
            quantiles = (rankdata(pred_vals) - 1) / (len(pred_vals) - 1)
            matched_embedding[:, i] = np.interp(quantiles, np.linspace(0, 1, len(layout_vals)), layout_vals)
        
        # שמירה לדיאגנוסטיקה
        self.matched_embedding = matched_embedding
        
        # 4. הכנת המרחב המנורמל (Weighted Distance)
        # אנחנו מחלקים בסטיית התקן כדי שוקטור 2 ו-3 יקבלו חשיבות שווה לוקטור 1
        layout_std = np.std(self.layout_graph.embedding[:, :matched_embedding.shape[1]], axis=0)
        L_weighted = self.layout_graph.embedding[:, :matched_embedding.shape[1]] / (layout_std + 1e-9)
        P_weighted = matched_embedding / (layout_std + 1e-9)
        
        # 5. חישוב מרחקים ומציאת שכנים
        from scipy.spatial.distance import cdist
        dists = cdist(P_weighted, L_weighted, metric='euclidean')
        
        if k_neighbors > 1:
            # ממוצע של K שכנים
            closest_indices = np.argsort(dists, axis=1)[:, :k_neighbors]
            all_coords = np.array([self.layout_graph.random_points.iloc[idx][['x', 'y']].values 
                                for idx in closest_indices])
            avg_coords = all_coords.mean(axis=1)
            res = pd.DataFrame(avg_coords, columns=['x', 'y'], index=validation_indices)
        else:
            # שכן אחד בלבד - מוודאים שהאינדקס הוא חד-ממדי למניעת ה-ValueError
            closest_indices = np.argmin(dists, axis=1) # מחזיר מערך 1D
            res = self.layout_graph.random_points.iloc[closest_indices].copy()
            res.index = validation_indices
            
        return res



In [12]:
import numpy as np
from scipy.spatial.distance import cdist
from scipy.linalg import orthogonal_procrustes

class ProcrustesManifoldMatcher:
    def __init__(self, layout_graph, signal_graph):
        self.layout_graph = layout_graph
        self.signal_graph = signal_graph
        self.R = None
        self.mu_signal = None
        self.mu_layout = None

    def fit(self, signal_indices, layout_indices):
        """
        signal_indices & layout_indices: הרשימות המקוריות של הלייבלים (IDs).
        אנחנו מתרגמים אותם למיקומים פיזיים בתוך מטריצות ה-Embedding.
        """
        # 1. תרגום אינדקסים למיקומים פיזיים (Positions)
        # משתמשים ב-get_indexer כדי לדעת איפה ה-ID נמצא בתוך ה-Embedding הנוכחי
        sig_pos = self.signal_graph.signal_data.index.get_indexer(signal_indices)
        lay_pos = self.layout_graph.random_points.index.get_indexer(layout_indices)

        # 2. שליפת ה-Anchors מה-Embedding (NumPy)
        # שימוש במיקומים הפיזיים שמצאנו כדי למנוע IndexError
        S = self.signal_graph.embedding[sig_pos]
        L = self.layout_graph.embedding[lay_pos]

        # 3. Center Data
        self.mu_signal = S.mean(axis=0)
        self.mu_layout = L.mean(axis=0)
        S_centered = S - self.mu_signal
        L_centered = L - self.mu_layout

        # 4. Orthogonal Procrustes
        self.R, _ = orthogonal_procrustes(S_centered, L_centered)
        print(f"✅ Fitted Rigid Procrustes Alignment (Matched {len(sig_pos)} points)")

    def predict(self, signal_indices_to_predict):
        """
        signal_indices_to_predict: רשימת ה-IDs לחיזוי (למשל ה-Validation set).
        """
        # 1. תרגום האינדקסים של ה-Signal למיקומים פיזיים
        predict_sig_pos = self.signal_graph.signal_data.index.get_indexer(signal_indices_to_predict)
        
        # 2. שליפת ה-Embedding הגולמי והצמדתו (Alignment)
        S_raw = self.signal_graph.embedding[predict_sig_pos]
        S_aligned = (S_raw - self.mu_signal) @ self.R + self.mu_layout
        
        # 3. חישוב מרחקים (Nearest Neighbor)
        # cdist מחזירה מטריצה של (N_predict x N_layout)
        dists = cdist(S_aligned, self.layout_graph.embedding, metric='euclidean')
        
        # 4. מציאת האינדקס של הנקודה הקרובה ביותר
        # min_idx מייצג את המיקומים הפיזיים (שורות) בתוך ה-layout_graph
        min_idx = np.argmin(dists, axis=1)
        
        # 5. החזרת הנקודות הפיזיות באמצעות iloc (כי min_idx הוא מיקום פיזי)
        predicted_points = self.layout_graph.random_points.iloc[min_idx].copy()
        
        # טיפ אופציונלי: החזרת האינדקס המקורי של ה-validation לצורך חישוב טעות קל
        predicted_points.index = signal_indices_to_predict
        
        return predicted_points

# functions- Load signals data functions

In [29]:
def build_wifi_fingerprint_df(site: str,
                              floor: int,
                              data_root: str,      # root folder containing train/test .txt files
                              metadata_root: str,  # where floor plan JSON + images are
                              split: str = "train"):
    """
    Build a WiFi fingerprinting DataFrame for a given site & floor,
    by aligning WiFi scans to nearest ground-truth/step positions (x, y).
    """

    # 1. find all trace files for that site & floor
    floor_str = f"F{floor}" if not str(floor).upper().startswith("F") else str(floor).upper()
    folder = Path(data_root) / split / site / floor_str
    txt_files = list(folder.glob("*.txt"))
    if not txt_files:
        raise FileNotFoundError(f"No .txt files found in {folder}")

    rows = []
    all_bssids = set()

    for fpath in txt_files:
        data = read_data_file(str(fpath))
        step_positions = compute_step_positions(data.acce, data.ahrs, data.waypoint)
        wifi_data = data.wifi  # shape: (n_scans, 5) — [timestamp, ..., bssid, rssi]

        if wifi_data.size == 0:
            continue

        # group by unique scan‑timestamp blocks
        for wifi_ds in split_ts_seq(wifi_data, np.unique(wifi_data[:,0].astype(float))):
            ts = float(wifi_ds[0,0])
            # find nearest step position by time
            diffs = np.abs(step_positions[:,0] - ts)
            idx = np.argmin(diffs)
            x, y = step_positions[idx,1], step_positions[idx,2]

            # build a dict: bssid → rssi (mean if multiple within same scan)
            bssid_to_rssi = {}
            for row in wifi_ds:
                bssid = row[2]
                rssi = int(row[3])
                if bssid in bssid_to_rssi:
                    # average if multiple
                    bssid_to_rssi[bssid] = (bssid_to_rssi[bssid] + rssi) / 2
                else:
                    bssid_to_rssi[bssid] = rssi
                all_bssids.add(bssid)

            # append row
            record = {"x": x, "y": y}
            record.update(bssid_to_rssi)
            rows.append(record)

    # 2. build DataFrame
    df = pd.DataFrame(rows)
    df.fillna(-999, inplace=True)  # for missing APs

    print(f"Built fingerprint DF shape: {df.shape} (rows: scans, cols: x,y + {len(all_bssids)} APs)")

    return df

def clean_wifi_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply cleaning similar to your load_filtered_wifi_data:
    - drop columns with all -999
    - drop rows where all signal columns are -999
    - replace -999 → -100
    """
    df = df.copy()
    # drop columns with all -999 (excluding x,y)
    signal_cols = [c for c in df.columns if c not in ("x","y")]
    df = df.drop(columns=[c for c in signal_cols if (df[c] == -999).all()])

    # drop rows where all signal cols are -999
    signal_cols = [c for c in df.columns if c not in ("x","y")]
    mask_all_missing = (df[signal_cols] == -999).all(axis=1)
    df = df.loc[~mask_all_missing].copy()

    # replace -999 with -100
    df[signal_cols] = df[signal_cols].replace(-999, -100)

    return df

def split_data_to_signals_locations(df):
    signals_locations_df = df.loc[:, ["x", "y"]]
    signals_df = df.drop(columns=["x", "y"])
    return signals_locations_df, signals_df

# functions- on area points

In [ ]:
def slice_points_near_signals(layout_points, signal_points, max_dist=3.0):
    """
    Keep only layout_graph points that have a signal point nearby.

    layout_points: (N,2) array of layout_graph.random_points (meters)
    signal_points: (M,2) array of signal positions (meters)
    max_dist: distance threshold (e.g., 3 meters)

    Returns:
        filtered_layout_points
        mask (boolean mask of kept points)
    """

    nbrs = NearestNeighbors(n_neighbors=1, algorithm='kd_tree').fit(signal_points)
    distances, _ = nbrs.kneighbors(layout_points)

    mask = distances[:, 0] <= max_dist
    return layout_points[mask], mask

# functions - plot

In [ ]:
def plot_heatmap_signals_data(site,floor, df_clean, width_meter, height_meter, floor_img):
    # To plot a heatmap (e.g. WiFi count or mean RSSI) : WiFi count
    heat_positions = df_clean[['x','y']].values
    heat_values = (df_clean.drop(columns=['x','y']) != -100).sum(axis=1).values

    visualize_heatmap(
        heat_positions, heat_values,
        str(floor_img), width_meter, height_meter,
        colorbar_title="WiFi count",
        title=f"WiFi Sum heatmap — site {site}, floor {floor}",
        show=True
    )

def plot_calibrated_area_points_and_signals_locations(signals_locations_df, calibrated_points):
    plt.figure(figsize=(8, 8))
    plt.scatter(signals_locations_df["x"], signals_locations_df["y"], s=2, color="red", label="signals (meters)")
    plt.scatter(calibrated_points[:, 0], calibrated_points[:, 1], s=2, color="blue", label="layout_graph points (calibrated from pixels to meters)")
    plt.legend()
    plt.title("Overlay: Calibrated layout_graph points + signal positions")
    plt.xlabel("x (meters)")
    plt.ylabel("y (meters)")
    plt.grid(True)
    plt.axis("equal")
    plt.show()

def plot_singals_and_nearest_area_points(layout_graph, filtered_pts, signals_locations_df):
    plt.figure(figsize=(8,8))

    plt.scatter(layout_graph.random_points.iloc[:,0], 
                layout_graph.random_points.iloc[:,1], 
                s=5, c='gray', alpha=0.2, label='all layout points')

    plt.scatter(filtered_pts.iloc[:,0], filtered_pts.iloc[:,1], 
                s=8, c='blue', alpha=0.9, label='kept (near signals)')

    plt.scatter(signals_locations_df.iloc[:,0], signals_locations_df.iloc[:,1], 
                s=5, c='red', label='signals')

    plt.legend()
    plt.axis('equal')
    plt.show()

def plot_labeled_points_over_all_points(random_points_with_labeled_signals, labeled_indices):
    rnd_pnts_indx = random_points_with_labeled_signals.index[~random_points_with_labeled_signals.index.isin(labeled_indices)]
    # Assuming df has columns: 'x', 'y', 'signal'
    plt.figure(figsize=(8, 6))
    plt.scatter(random_points_with_labeled_signals.loc[rnd_pnts_indx]['x'], random_points_with_labeled_signals.loc[rnd_pnts_indx]['y'],c="green", s=10)
    plt.scatter(random_points_with_labeled_signals.loc[labeled_indices]['x'], random_points_with_labeled_signals.loc[labeled_indices]['y'],c="blue", s=10)
    # Add colorbar to show signal values
    # plt.colorbar(sc, label='Signal Strength')
    # Axis labels and title
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('random points (green) and signal labeled points (blue)')
    plt.grid(True)
    plt.axis('equal')
    plt.tight_layout()
    plt.show()

def plot_points_colored_by_eigen_vectors(locations_df, graph_inst, data_type):
    fig,axs = plt.subplots(1,EMBEDDING_DIM,figsize = (20,5))
    for i,ax in enumerate(axs.flatten()):
        ax.scatter(locations_df["x"],locations_df["y"],c = graph_inst.embedding[:,i],s = 14,cmap = 'cividis')
    # Add a title to the entire figure
    fig.suptitle(f"{data_type} data colored by the first eigen vectors of the {data_type} graph", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Leave space for the suptitle
    plt.show()

def plot_predicted_points_and_real_locations(validation_true_real_locations, predicted_locations_validation, model_type, output_path, floor,site):
    # Plot the predicted and true locations for the validation data on the layout
    plt.figure(figsize=(8, 8))
    # Plot the true locations of the validation points
    plt.scatter(validation_true_real_locations.iloc[:, 0], validation_true_real_locations.iloc[:, 1], color='blue', s=100, label="True Validation Locations", marker='o', edgecolor='black')
    # Compute deltas
    dx = validation_true_real_locations.iloc[:, 0] - predicted_locations_validation[:, 0]
    dy = validation_true_real_locations.iloc[:, 1] - predicted_locations_validation[:, 1]
    # Draw arrows
    plt.quiver(predicted_locations_validation[:, 0], predicted_locations_validation[:, 1], dx, dy, angles='xy', scale_units='xy', scale=1, color='lightblue', width=0.005)
    # Plot the predicted locations of the validation points
    plt.scatter(predicted_locations_validation[:, 0], predicted_locations_validation[:, 1], color='red', s=100, label="Predicted Validation Locations", marker='X', edgecolor='black')
    plt.legend()
    plt.title(f"Predicted vs True Locations for Validation Data - {model_type}")
    
    save_dir = os.path.join(output_path, "sites_prediction", str(site), str(floor))
    full_file_path = os.path.join(save_dir, f"Predicted_vs_True_{model_type}.png")
    plt.savefig(full_file_path, dpi=300, bbox_inches='tight') 
    # plt.show()

def plot_points_colored_calibread_signal_eigen_vec_vs_colored_layout_random_eigen_vec(layout_graph, calibrated_phi_s, signals_locations_df,model_type, output_path, floor, site):
    fig,axs = plt.subplots(2,EMBEDDING_DIM,figsize = (10,5))
    for i,ax in enumerate(axs.flatten()[0:EMBEDDING_DIM]):
        ax.scatter(layout_graph.random_points["x"],layout_graph.random_points["y"],c = layout_graph.embedding[:,i],s = 14,cmap = 'cividis')

    for i,ax in enumerate(axs.flatten()[EMBEDDING_DIM:EMBEDDING_DIM*2]):
        ax.scatter(signals_locations_df["x"],signals_locations_df["y"],c = calibrated_phi_s[:,i],s = 14,cmap = 'cividis')
    # Add a title to the entire figure
    fig.suptitle("sampled_points colored by: \n first row: the first eigen-vectors of the layout (area) graph \n second row: eigenvectors of calibrated_phi_s", fontsize=16)

    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Leave space for the suptitle

    save_dir = os.path.join(output_path, "sites_prediction", str(site), str(floor))
    full_file_path = os.path.join(save_dir, f"colored_calibread_signal_eigen_vec_vs_colored_layout_random_eigen_vec_{model_type}.png")
    plt.savefig(full_file_path, dpi=300, bbox_inches='tight')     
    # plt.show()
    

# functions - filter routers 

In [ ]:
def filter_high_variance_routers(signals_df, top_n=100):
    """
    שומרת רק את הראוטרים שמשתנים הכי הרבה לאורך כל הדגימות.
    """
    # חישוב שונות לכל עמודה (ראוטר)
    # הערה: אם יש ערכי 100- קבועים, השונות שלהם תהיה 0, וזה מצוין.
    router_variances = signals_df.var(axis=0)
    
    # בחירת הראוטרים עם השונות הגבוהה ביותר
    top_routers = router_variances.sort_values(ascending=False).head(top_n).index
    
    filtered_df = signals_df[top_routers]
    
    print(f"✅ סינון הושלם: נשארו {top_n} ראוטרים מתוך {signals_df.shape[1]}")
    print(f"טווח שונות שנבחר: {router_variances[top_routers].min():.2f} - {router_variances[top_routers].max():.2f}")
    
    return filtered_df

# הרצה: נסי להתחיל ב-100 או 150 ראוטרים (מתוך ה-400+ שיש לך)
signals_df_filtered = filter_high_variance_routers(df_clean, top_n=200)

# run Microsoft Data 

In [16]:
# The magic command that tells Jupyter/IPython to render Matplotlib figures inline
%matplotlib inline

In [ ]:
site = "5a0546857ecc773753327266"
floor = 2
wifi_root = r"C:\Users\Noa\Documents\GitHub\indoor_localization_research\Data\Microsoft data\wifi_features"
metadata_root = r"C:\Users\Noa\Documents\GitHub\indoor_localization_research\Data\Microsoft data\Data\metadata"
output_path = r"C:\Users\Noa\Documents\Thesis\Outputs"

data_root = r"C:/Users/Noa/Documents/GitHub/indoor_localization_research" 
metadata_root = r"C:/Users/Noa/Documents/GitHub/indoor_localization_research/metadata"
# fstr = f"F{floor}" if not str(floor).upper().startswith("F") else str(floor).upper()
# img_path = next((Path(metadata_root)/site/fstr).glob("floor_image.*"))

In [22]:
def define_param_floor(site,floor, metadata_root):
    # fstr = f"F{floor}" if not str(floor).upper().startswith("F") else str(floor).upper()
    # img_path = next((Path(metadata_root)/site/fstr).glob("floor_image.*"))
    floor_info_path = Path(metadata_root)/site/f"F{floor}" / "floor_info.json"
    with open(floor_info_path) as f:
        inf = json.load(f)
    width_meter, height_meter = inf["map_info"]["width"], inf["map_info"]["height"]
    floor_img = Path(metadata_root)/site/f"F{floor}" / "floor_image.png"
    return floor_img, width_meter, height_meter

In [ ]:
def sample_points_from_img_create_layout_graph(floor, site, metadata_root):
    layout_graph = PhysicalLayoutGraph(floor, site, metadata_root, num_points=NUM_POINTS, embedding_dim=EMBEDDING_DIM, k=7)
    return layout_graph
    dist_matrix = compute_shortest_paths(layout_graph.G, layout_graph.random_points, layout_graph.circles)
    file_path_name = fr"C:\Users\Noa\Documents\GitHub\indoor_localization_research\Data\dist_mat_{site}_{floor}_{NUM_POINTS}_num_points.pkl"
    with open(file_path_name, "wb") as f:
        pickle.dump({'dist_matrix': dist_matrix, 'layout_graph': layout_graph}, f)
    pd.to_pickle(dist_matrix, fr"C:\Users\Noa\Documents\GitHub\indoor_localization_research\Data\distance_matrix_{NUM_POINTS}points_{NUM_RECIVERS}recivers.pkl")


def save_layout_graph_inst(floor, site, metadata_root, layout_graph, points_file_name, output_path):
    # 2. Construct the new folder path: Data\sites_prediction\SITE\FLOOR
    # We use os.path.join for safe path construction across different OSs
    save_dir = os.path.join(output_path, "sites_prediction", str(site), str(floor))
    # 3. Create the directory (and any missing parent directories)
    # exist_ok=True prevents errors if the folder already exists
    os.makedirs(save_dir, exist_ok=True)
    # 4. Construct the filename
    # 5. Combine directory and filename
    full_file_path = os.path.join(save_dir, points_file_name)
    # 6. Save the file
    with open(full_file_path, "wb") as f:
        pickle.dump({'layout_graph': layout_graph}, f)

def calibrate_points_to_meters(points, image_shape, width_meter, height_meter):
    """
    Convert (y,x) pixel coordinates to (x,y) in meters.
    Parameters:
        points: ndarray of shape (N,2), with (y,x) pixel coordinates.
        image_shape: tuple (H, W) of the image (in pixels).
        width_meter: width of the map in meters (real-world).
        height_meter: height of the map in meters (real-world).
    Returns:
        calibrated: ndarray of shape (N,2), real-world coordinates in meters.
    """
    H, W = image_shape
    ys, xs = points[:, 0], points[:, 1]

    # Flip y-axis to match top-down view, then scale
    xs_m = xs / W * width_meter
    ys_m = (H - ys) / H * height_meter  # flip y

    return np.column_stack((xs_m, ys_m))  # shape: (N, 2)

In [ ]:
def read_preprocessed_signals_data(data_root, site, floor):
    out_path = Path(data_root) / "preprocessed_data_thesis" / f"wifi_fp_{site}_{floor}.csv"
    df_clean = pd.read_csv(out_path)
    #plot_heatmap_signals_data(site,floor,df_clean, width_meter, height_meter, floor_img)
    df_clean = df_clean.sample(frac=1, random_state=42)
    df_clean = df_clean.reset_index(drop=True)

def split_indices_to_train_validation( n_signals, n_labeled = 50, n_validation = 200):
    signals_indices = np.arange(n_signals)
    rng = np.random.default_rng(seed=42)
    combined_sample = rng.choice(signals_indices, size=n_labeled + n_validation, replace=False)
    labeled_indices = combined_sample[:n_labeled]
    validation_indices = combined_sample[n_labeled:]
    return labeled_indices, validation_indices

def create_points_df_to_layout_graph(all_points, labeled_indices, n_random_points_sampled_img):
    rnd_pnt = all_points.iloc[-n_random_points_sampled_img:]
    lbld_sig = all_points.iloc[labeled_indices]
    random_points_with_labeled_signals = pd.concat([rnd_pnt, lbld_sig], axis=0)
    # NOTE - those points are the random points from the image plus the labeled signals
    layout_graph.random_points = random_points_with_labeled_signals
    # plot_labeled_points_over_all_points(random_points_with_labeled_signals, labeled_indices)

In [ ]:
N_LABELED = 5
N_VALIDATION = 200

In [ ]:
floor_img, width_meter, height_meter = define_param_floor(site,floor, metadata_root)
layout_graph = sample_points_from_img_create_layout_graph(floor, site, metadata_root)
save_layout_graph_inst(floor, site, metadata_root, layout_graph, f"NUMPOINTS_{NUM_POINTS}_inst_k={layout_graph.k}_points.pkl", output_path)
# Assume layout_graph.random_points is in (y, x)
image_shape = layout_graph.polygon_mask.shape  # (H, W)
layout_graph.random_points = calibrate_points_to_meters(layout_graph.random_points,image_shape=image_shape,width_meter=width_meter,height_meter=height_meter)

df_clean = read_preprocessed_signals_data(data_root, site, floor)
signals_locations_df, signals_df = split_data_to_signals_locations(df_clean)
layout_graph.random_points = pd.DataFrame(layout_graph.random_points, columns=["x","y"])
layout_graph.random_points, mask = slice_points_near_signals(layout_graph.random_points,signals_locations_df,max_dist=2.0)   # tune (2-5m)
# plot_singals_and_nearest_area_points(layout_graph, layout_graph.random_points, signals_locations_df)

all_points = pd.concat([signals_locations_df,layout_graph.random_points] , ignore_index=True) # concat all signals with all random points to 1 df with reseted indeces
# Add labeled signals locations to the random points from layout_graph
labeled_indices, validation_indices = split_indices_to_train_validation(len(signals_locations_df),  N_LABELED, N_VALIDATION)
layout_graph.random_points = create_points_df_to_layout_graph(all_points, labeled_indices, len(layout_graph.random_points))

signal_graph = SignalGraph(signals_df, embedding_dim=EMBEDDING_DIM, k_nearest=7)
signal_graph._compute_embedding(EMBEDDING_DIM)
# plot_points_colored_by_eigen_vectors(signals_locations_df, signal_graph, data_type="signals")
layout_graph._build_graph()
layout_graph._compute_adjacency()
layout_graph._compute_embedding()
save_layout_graph_inst(floor, site, metadata_root, layout_graph, f"graph_with_n_points={NUM_POINTS}_k={layout_graph.k}_inst.pkl", output_path)
# plot_points_colored_by_eigen_vectors(layout_graph.random_points, layout_graph, data_type="layout")

matcher_ssl = ManifoldMatcherSSL()
phi_s, centered_A, C, calib_phi_s = matcher_ssl.fit(layout_graph, signal_graph, labeled_indices)
validation_signal_data = signals_df.iloc[validation_indices]
validation_true_real_locations = signals_locations_df.iloc[validation_indices]
predicted_locations_validation_SSL = matcher_ssl.predict(validation_indices)
plot_predicted_points_and_real_locations(validation_true_real_locations, predicted_locations_validation_SSL,"SSL", output_path,floor,site)

manifold_mm = ManifoldMatcher(layout_graph, signal_graph)
phi_S, C_mat, calibrated_phi_s_labeled = manifold_mm.fit(labeled_indices, EMBEDDING_DIM)
calibrated_phi_s = signal_graph.embedding @ C_mat
plot_points_colored_calibread_signal_eigen_vec_vs_colored_layout_random_eigen_vec(layout_graph, calibrated_phi_s, signals_locations_df, "Manifold Matcher", output_path, floor, site)
predicted_locations_validation_manifold_matching = manifold_mm.predict(validation_indices, EMBEDDING_DIM)
plot_predicted_points_and_real_locations(validation_true_real_locations, predicted_locations_validation_manifold_matching,"Manifold Matcher", output_path,floor,site)

